<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/03_gov_contracts_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Government-Contracts Signal

This is the project's second alternative-data signal, and the first of the three that
join the congressional signal in the composite. It follows the same `BaseSignal`
contract, so the backtester from notebook `02` and the composite scorer treat it
identically to the congress signal — a $0$–$100$ score per ticker, comparable across
datasets.

### Signal design

The thesis: a company winning federal contracts, especially an accelerating flow of
them across several agencies, has a fundamental tailwind that the market may price in
slowly. The score blends four features, each a testable piece of that idea:

| Feature | Hypothesis |
|---|---|
| `award_value` | More recent award dollars signal a stronger tailwind |
| `acceleration` | Award flow rising above its prior run-rate is newly informative |
| `n_awards` | Repeated distinct wins indicate durable demand, not one lucky bid |
| `agency_breadth` | Demand across many agencies is broader than a single-buyer relationship |

Awards key on the announcement date, public when it posts, so the signal carries no
look-ahead — the same discipline the congress signal follows.


## Setup

Defaults to mock data, so it runs with no key. Set `USE_LIVE = True` for live Quiver
government-contract data.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data

Working in: /content/quant-equity-research


## Install the package

This notebook's code lives in the repository under `src/quant_research/`; the clone above brought it in. The cell below installs the package in editable mode and puts it on the session path.

In [2]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.2.0


## Construct the signal

We compute the signal as of today over a $90$-day lookback, treating the most recent
$30$ days as the window whose flow is compared against the prior run-rate. `compute`
returns a DataFrame indexed by ticker, sorted by score, with the features behind it.


In [3]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.govcontracts import GovContractsSignal
import pandas as pd

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = GovContractsSignal(client, lookback_days=90, recent_days=30)

scores = signal.compute()
print(f"{len(scores)} tickers scored | data: {'live' if USE_LIVE else 'mock'}")

disp = scores.head(12).copy()
disp["award_$M"] = (disp["award_value"] / 1e6).round(1)
disp["accel_$M"] = (disp["acceleration"] / 1e6).round(1)
disp[["score", "award_$M", "accel_$M", "n_awards", "agency_breadth"]]

260 tickers scored | data: live


,score,award_$M,accel_$M,n_awards,agency_breadth
ticker,,,,,
DELL,100.0,200.3,200.3,39,13
F,78.9,236.2,-85.3,4567,1
GD,71.4,204.9,44.5,27,10
PLTR,70.5,269.9,-7.8,8,4
MDLN,68.0,258.9,-2.0,100,3
TPC,58.0,225.3,10.1,5,1
BAH,53.0,231.9,-109.8,35,10
TXT,42.4,217.9,-109.0,8,5
DNOW,31.0,3.3,-0.6,3713,3


### Reading the output

`score` is the $0$–$100$ conviction reading. The `*_n` columns are the features
normalized across the cross-section — the actual inputs to the weighted score, so each
score is fully explainable.


In [4]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,award_value_n,acceleration_n,n_awards_n,agency_breadth_n
ticker,,,,,
DELL,100.0,0.742,1.000,0.008,1.000
F,78.9,0.875,0.079,1.000,0.000
GD,71.4,0.759,0.498,0.006,0.750
PLTR,70.5,1.000,0.329,0.002,0.250
MDLN,68.0,0.959,0.348,0.022,0.167
TPC,58.0,0.835,0.387,0.001,0.000
BAH,53.0,0.859,0.000,0.007,0.750
TXT,42.4,0.807,0.003,0.002,0.333


### Top names, shaded by acceleration

The chart shades each candidate by acceleration, so you can see whether a high score
comes from a large but steady award base or from a genuine pickup in recent flow.


In [5]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="acceleration_n", color_continuous_scale="Oranges",
             labels={"acceleration_n": "acceleration"},
             title="Gov-contracts signal — top 15 (shading = acceleration)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## Reading it honestly

A few caveats specific to this signal:

- **Awards are lumpy.** A single large contract can dominate a ticker's score for a
  quarter, so acceleration and agency breadth matter as much as raw dollars — a broad,
  rising flow is more durable than one headline award.
- **A contract is not booked revenue.** Announced award value is a ceiling that pays
  out over time and can be reduced or cancelled, so treat the dollar figures as a
  signal of momentum rather than a fundamental estimate.
- **No enrichment needed here.** Unlike the congress signal, this one needs no net
  worth or committee data — the award flow speaks for the company directly.

The same event-study engine from notebook `02` validates this signal without changes,
since it depends only on the `BaseSignal` contract. We'll run that once the full signal
set is in place.


## Recap and next

The second signal is built on the shared contract: a transparent government-contract
score driven by award level, acceleration, repeat wins, and agency breadth, with the
same disclosure-date discipline as the congress signal.

Next come the lobbying and off-exchange signals on the same pattern, then the composite
scorer that blends all four into one ranking, and a combined backtest of the composite
through the `02` engine. The dashboard notebook then surfaces today's candidates with
the historical edge behind each.
